# FrameAI — Style Annotator (Qwen2-VL-7B)

Uses **Qwen2-VL-7B-Instruct** (4-bit) on a T4 GPU to classify eyeglass frame style
for the 129 records that couldn't be auto-tagged from their image URL.

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Upload `no_style_records.json` using the 📁 Files panel on the left
3. Runtime → Run all
4. `style_predictions.csv` downloads automatically at the end


In [ ]:
# Cell 1 — Install
!pip install transformers torch pillow requests tqdm qwen-vl-utils accelerate bitsandbytes -q

In [ ]:
# Cell 2 — Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU! Set Runtime → Change runtime type → T4 GPU."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3 — Load Qwen2-VL-7B-Instruct (4-bit, ~6 GB VRAM)
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading model (~15 GB download on first run, ~6 GB VRAM after quantization)...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(f"Model ready. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 4 — Classifier
from qwen_vl_utils import process_vision_info
from PIL import Image

VALID_STYLES = [
    "rectangular", "round", "oval", "square",
    "cat-eye", "aviator", "wayfarer", "browline",
    "rimless", "geometric", "oversized",
]
STYLE_LIST = ", ".join(VALID_STYLES)

PROMPT = (
    f"These are eyeglasses on a white background. "
    f"What is the frame style? "
    f"Choose exactly one from this list: {STYLE_LIST}. "
    f"Reply with just the style name, nothing else."
)

# Handle slight variations the model might return
ALIASES = {
    "rectangle": "rectangular",
    "cat eye": "cat-eye",
    "cateye": "cat-eye",
    "cat_eye": "cat-eye",
    "pilot": "aviator",
    "teardrop": "aviator",
    "brow line": "browline",
    "brow-line": "browline",
    "frameless": "rimless",
    "semi-rimless": "rimless",
    "angular": "square",
    "hexagonal": "geometric",
    "octagonal": "geometric",
}


def classify(image: Image.Image) -> tuple[str | None, str]:
    """Returns (matched_style_or_None, raw_model_answer)."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": PROMPT},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)

    raw = processor.batch_decode(
        out[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )[0].strip().lower()

    # Match to valid style
    if raw in VALID_STYLES:
        return raw, raw
    if raw in ALIASES:
        return ALIASES[raw], raw
    for style in VALID_STYLES:
        if style in raw:
            return style, raw
    for alias, style in ALIASES.items():
        if alias in raw:
            return style, raw
    return None, raw


print("Classifier ready.")

In [ ]:
# Cell 5 — Load records
import json
from pathlib import Path

DATA_FILE = "no_style_records.json"
assert Path(DATA_FILE).exists(), "Upload no_style_records.json via the Files panel first."

records = json.loads(Path(DATA_FILE).read_text())
print(f"Loaded {len(records)} records to classify")

In [ ]:
# Cell 6 — Image downloader
import requests
from io import BytesIO

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    ),
    "Accept": "image/webp,image/apng,image/*,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}


def fetch_image(url: str) -> Image.Image | None:
    referer = "https://www.lenskart.com/" if "lenskart" in url else "https://www.titaneyeplus.com/"
    try:
        resp = requests.get(url, headers={**HEADERS, "Referer": referer}, timeout=15)
        resp.raise_for_status()
        return Image.open(BytesIO(resp.content)).convert("RGB")
    except Exception:
        return None


# Quick test — download + classify one image end-to-end
test_url = next(
    (r.get("product_image_url") or r.get("image_url") for r in records
     if r.get("product_image_url") or r.get("image_url")),
    None,
)
if test_url:
    img = fetch_image(test_url)
    if img:
        style, raw = classify(img)
        print(f"Download OK + model working.")
        print(f"  Raw answer : '{raw}'")
        print(f"  Matched    : '{style}'")
        display(img.resize((200, 133)))
    else:
        print(f"WARNING: image download failed — {test_url}")
        print("CDN may be blocking Colab. Affected records will be marked 'download_failed'.")

In [ ]:
# Cell 7 — Run classification on all records
from tqdm.notebook import tqdm

results = []
n_classified = n_no_url = n_failed = n_unrecognised = 0

for rec in tqdm(records, desc="Classifying"):
    frame_id = rec["frame_id"]
    name     = rec.get("name", "")
    url      = rec.get("product_image_url") or rec.get("image_url") or ""

    if not url:
        results.append({"frame_id": frame_id, "name": name,
                        "predicted_style": "", "raw_answer": "no_url", "status": "no_url"})
        n_no_url += 1
        continue

    image = fetch_image(url)
    if image is None:
        results.append({"frame_id": frame_id, "name": name,
                        "predicted_style": "", "raw_answer": "download_failed", "status": "download_failed"})
        n_failed += 1
        continue

    style, raw = classify(image)
    if style is None:
        n_unrecognised += 1
    else:
        n_classified += 1
    results.append({"frame_id": frame_id, "name": name,
                    "predicted_style": style or "", "raw_answer": raw,
                    "status": "ok" if style else "unrecognised"})

print(f"\nDone — {len(results)} records processed")
print(f"  Classified:      {n_classified}")
print(f"  Unrecognised:    {n_unrecognised}")
print(f"  Download failed: {n_failed}")
print(f"  No image URL:    {n_no_url}")

In [ ]:
# Cell 8 — Preview results
import pandas as pd

df = pd.DataFrame(results)
print("Style distribution (classified only):")
print(df[df.status == "ok"]["predicted_style"].value_counts().to_string())
print()
print("Unrecognised answers (need manual check):")
bad = df[df.status == "unrecognised"]
if len(bad):
    print(bad[["name", "raw_answer"]].to_string())
else:
    print("  None!")
print()
display(df.head(15))

In [ ]:
# Cell 9 — Save and download
import csv
from google.colab import files

OUT_FILE = "style_predictions.csv"
with open(OUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f, fieldnames=["frame_id", "name", "predicted_style", "raw_answer", "status"]
    )
    writer.writeheader()
    writer.writerows(results)

print(f"Saved {len(results)} rows → {OUT_FILE}")
files.download(OUT_FILE)